# 04 · IBM Quantum Hardware Results from Recorded Counts

This notebook reproduces the IBM Quantum hardware results of the benchmark offline, from the raw counts that the `ibm_fez` jobs returned and that are versioned in the repository. It needs no IBM account and makes no IBM Quantum call: the counts are re-read, the hardware test kernel is rebuilt and calibrated on training pairs, the already trained quantum-kernel SVM classifies the test set from it, and every number is checked against the published results and the protocol. It also checks the published replay file and, optionally, re-runs the noisy rehearsal on a local fake backend.

**Inputs**
- `reports/qpu-evidence/full-ibm_fez-20261025T094010Z/` and `reports/qpu-evidence/pilot-ibm_fez-20261025T092140Z/` (`run.json`, `counts-<job>.json`)
- `configs/benchmark-split.json`, `configs/feature-scaler.json`, `data/processed/benchmark-sample.csv`
- `artifacts/benchmark/` (trained models, training angles, ideal test kernel, `evaluation-simulation.json`) from the earlier notebooks
- `reports/benchmark-results.json`, `reports/benchmark-replay.json`, `docs/benchmark-protocol.md` (published values to compare against)

**Outputs**
- `artifacts/benchmark/qpu-runs/<run_id>/`: copies of the recorded evidence plus `evaluation.json` and, for the full run, `test-kernel-raw.npy` and `test-kernel-calibrated.npy`
- No published file under `reports/` or `docs/` is modified.

Run after: `03` (simulation evaluation; it writes `artifacts/benchmark/evaluation-simulation.json`).

Every number is labeled by source: **exact simulation** (statevector), **IBM hardware, uncorrected**, **IBM hardware, calibrated**, **noisy simulation** (local fake backend), or **replay** (recorded predictions).

In [1]:
import hashlib
import json
import os
import shutil
import sys
import warnings
from datetime import datetime
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src").is_dir() and (p / "configs").is_dir()
)
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd

from src.evaluation import inputs as benchmark_inputs
from src.evaluation import replay
from src.evaluation.metrics import evaluate
from src.quantum import hardware, qpu_run
from src.quantum import kernel as qkernel

FULL_RUN = "full-ibm_fez-20261025T094010Z"
PILOT_RUN = "pilot-ibm_fez-20261025T092140Z"
EVIDENCE = Path("reports/qpu-evidence")
RUNS_DIR = qpu_run.RUNS_DIR.relative_to(ROOT)  # artifacts/benchmark/qpu-runs
TOL = 1e-9


def rel(path) -> str:
    # Paths are shown relative to the repository root only.
    return Path(path).resolve().relative_to(ROOT).as_posix()


def read(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


print("Repository root found; working directory set to it.")
print("Recorded evidence:", sorted(p.name for p in EVIDENCE.iterdir() if p.is_dir()))

Repository root found; working directory set to it.
Recorded evidence: ['full-ibm_fez-20261025T094010Z', 'pilot-ibm_fez-20261025T092140Z']


## 1. The hardware workload

The SVM is trained classically on an exact simulated 24 x 24 kernel. Hardware only measures the kernel values the trained model needs to classify the test set, plus training-side controls. Each kernel value $K(x, y) = |\langle 0000 | U(y)^\dagger U(x) | 0000 \rangle|^2$ is estimated by one compute-uncompute circuit as the fraction of shots that return `0000`.

- **288 test pairs**: each of the 12 test examples against each of the 24 training examples.
- **24 calibration pairs**: off-diagonal training pairs drawn with seed 42 from the 276 available. Their exact values are known, so they fit the calibration.
- **24 diagonal controls**: each training example against itself (ideal value 1). These circuits simplify almost to the identity, so they mainly measure readout error.

All 336 circuits ran with 1,024 shots in one SamplerV2 job on `ibm_fez`, after ISA transpilation with optimization level 1 and transpiler seed 42. The recorded run record holds the plan, the transpilation statistics, the job ID, and IBM's usage metrics. It contains no credentials.

In [2]:
full_record = read(EVIDENCE / FULL_RUN / "run.json")
pilot_record = read(EVIDENCE / PILOT_RUN / "run.json")
job = full_record["jobs"][0]
metrics = job["metrics"]
t = full_record["transpilation"]

plan_kinds = {
    kind: sum(spec["kind"] == kind for spec in full_record["plan"])
    for kind in ("test", "calibration", "diagonal")
}
print("Plan:", plan_kinds, "total", len(full_record["plan"]))
overview = {
    "run_id": full_record["run_id"],
    "backend": full_record["backend"],
    "sampler": full_record["sampler"],
    "shots per circuit": full_record["shots"],
    "jobs": len(full_record["jobs"]),
    "job_id": job["job_id"],
    "job status": job["status"],
    "SDK versions (IBM metrics)": metrics["qiskit_version"],
    "optimization level / transpiler seed": f"{t['optimization_level']} / {t['seed_transpiler']}",
    "depth min / mean / max": f"{t['depth']['min']} / {t['depth']['mean']:.1f} / {t['depth']['max']}",
    "two-qubit gates min / mean / max": f"{t['two_qubit_gates']['min']} / {t['two_qubit_gates']['mean']:.2f} / {t['two_qubit_gates']['max']}",
    "physical qubit layouts": str(t["distinct_physical_layouts"]),
}
pd.Series(overview, name="IBM hardware run (recorded)").to_frame()

Plan: {'test': 288, 'calibration': 24, 'diagonal': 24} total 336


,IBM hardware run (recorded)
run_id,full-ibm_fez-20261025T094010Z
backend,ibm_fez
sampler,v2
shots per circuit,1024
jobs,1
job_id,dbessv7pc3tc739mbc6g
job status,DONE
SDK versions (IBM metrics),"qiskit_ibm_runtime-0.51.0,qiskit-2.6.1*,qiskit..."
optimization level / transpiler seed,1 / 42
depth min / mean / max,1 / 35.8 / 54


In [3]:
stamps = {
    key: datetime.fromisoformat(value.replace("Z", "+00:00"))
    for key, value in metrics["timestamps"].items()
}
usage = metrics["usage"]
timing = {
    "job created (UTC)": stamps["created"].isoformat(timespec="seconds"),
    "job running (UTC)": stamps["running"].isoformat(timespec="seconds"),
    "job finished (UTC)": stamps["finished"].isoformat(timespec="seconds"),
    "queue wait, created -> running (s)": round(
        (stamps["running"] - stamps["created"]).total_seconds(), 1
    ),
    "running -> finished, wall (s)": round(
        (stamps["finished"] - stamps["running"]).total_seconds(), 1
    ),
    "billed QPU time, qpu_charge_time_seconds (s)": usage["qpu_charge_time_seconds"],
    "resource units (Open Plan)": usage["value"],
    "circuit execution time (s)": metrics["circuits_execution_time_ns"] / 1e9,
}
pd.Series(timing, name="IBM-reported usage (recorded)").to_frame()

,IBM-reported usage (recorded)
job created (UTC),2026-10-25T09:40:12+00:00
job running (UTC),2026-10-25T09:40:13+00:00
job finished (UTC),2026-10-25T09:41:53+00:00
"queue wait, created -> running (s)",1.3
"running -> finished, wall (s)",100.0
"billed QPU time, qpu_charge_time_seconds (s)",92
resource units (Open Plan),1.526017
circuit execution time (s),89.187153


In [4]:
# Replication check: the recorded workload matches the protocol and the code's own plan.
data = benchmark_inputs.load()
assert plan_kinds == {"test": 288, "calibration": 24, "diagonal": 24}
assert full_record["plan"] == [vars(spec) for spec in qpu_run.full_plan(data)], (
    "recorded plan differs from the code's plan"
)
assert full_record["shots"] == hardware.SHOTS == 1024 and full_record["sampler"] == "v2"
assert (
    (t["optimization_level"], t["seed_transpiler"])
    == (hardware.OPTIMIZATION_LEVEL, hardware.SEED)
    == (1, 42)
)
assert full_record["inputs"] == pilot_record["inputs"] == data.input_hashes, (
    "runs used other frozen inputs"
)
assert full_record["status"] == pilot_record["status"] == "retrieved" and job["status"] == "DONE"
assert usage["qpu_charge_time_seconds"] == 92 and round(usage["value"], 2) == 1.53
assert round(metrics["circuits_execution_time_ns"] / 1e9, 1) == 89.2
print(
    "OK: 336-circuit plan (288 test / 24 calibration / 24 diagonal) equals hardware.plan_pairs with seed 42"
)
print("OK: run inputs match the current split manifest and scaler hashes")
print("OK: usage 92 s billed, 1.53 resource units, 89.2 s execution")

OK: 336-circuit plan (288 test / 24 calibration / 24 diagonal) equals hardware.plan_pairs with seed 42
OK: run inputs match the current split manifest and scaler hashes
OK: usage 92 s billed, 1.53 resource units, 89.2 s execution


## 2. Copy the evidence and evaluate it offline

`qpu_run.evaluate_run` is the function behind `scripts/run_qpu.py evaluate`. It writes `evaluation.json` and kernel `.npy` files into the run directory, so the versioned evidence is first copied into the Git-ignored `artifacts/benchmark/qpu-runs/<run_id>/`. An existing copy is accepted only if its record and counts are byte-identical to the evidence. `results_path=None` keeps `reports/benchmark-results.json` untouched; the published file is only compared against.

In [5]:
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def copy_evidence(run_id):
    source, target = EVIDENCE / run_id, RUNS_DIR / run_id
    target.mkdir(parents=True, exist_ok=True)
    for file in sorted(source.glob("*.json")):
        destination = target / file.name
        if destination.exists():
            assert sha256(destination) == sha256(file), (
                f"{rel(destination)} differs from the versioned evidence"
            )
            status = "already present, identical"
        else:
            shutil.copy2(file, destination)
            status = "copied"
        print(f"{rel(file)} -> {rel(destination)} ({status}, sha256 {sha256(file)[:12]}...)")
    return target


full_dir = copy_evidence(FULL_RUN)
pilot_dir = copy_evidence(PILOT_RUN)

reports/qpu-evidence/full-ibm_fez-20261025T094010Z/counts-dbessv7pc3tc739mbc6g.json -> artifacts/benchmark/qpu-runs/full-ibm_fez-20261025T094010Z/counts-dbessv7pc3tc739mbc6g.json (copied, sha256 9b7a754a13cc...)
reports/qpu-evidence/full-ibm_fez-20261025T094010Z/run.json -> artifacts/benchmark/qpu-runs/full-ibm_fez-20261025T094010Z/run.json (copied, sha256 05949b4cf572...)
reports/qpu-evidence/pilot-ibm_fez-20261025T092140Z/counts-dbesk98m94ac738secsg.json -> artifacts/benchmark/qpu-runs/pilot-ibm_fez-20261025T092140Z/counts-dbesk98m94ac738secsg.json (copied, sha256 aaedfab850ef...)
reports/qpu-evidence/pilot-ibm_fez-20261025T092140Z/run.json -> artifacts/benchmark/qpu-runs/pilot-ibm_fez-20261025T092140Z/run.json (copied, sha256 80ecd33c01ed...)


In [6]:
full = qpu_run.evaluate_run(full_dir, data, results_path=None)
pilot = qpu_run.evaluate_run(pilot_dir, data, results_path=None)
assert "published" not in full and "published" not in pilot
print(
    "Wrote:",
    rel(full_dir / "evaluation.json"),
    rel(full_dir / "test-kernel-raw.npy"),
    rel(full_dir / "test-kernel-calibrated.npy"),
)
print("Wrote:", rel(pilot_dir / "evaluation.json"))

cal = full["calibration"]
hardware_summary = {
    "circuits": full["circuits"],
    "shots returned per circuit": str(full["shots_returned"]),
    "diagonal controls mean (ideal 1)": full["diagonal_controls_mean"],
    "calibration slope a": cal["a"],
    "calibration intercept c": cal["c"],
    "calibration RMSE": cal["rmse"],
    "calibrated test values clipped (of 288)": cal["clipped_test_values"],
    "test kernel error vs exact, uncorrected": full["test_kernel_error_vs_ideal"]["qpu"],
    "test kernel error vs exact, calibrated": full["test_kernel_error_vs_ideal"]["qpu_calibrated"],
    "error vs exact over all 336 circuits, uncorrected": full["mean_abs_kernel_error_vs_ideal"],
}
pd.Series(hardware_summary, name="IBM hardware, recomputed from counts").to_frame()

Wrote: artifacts/benchmark/qpu-runs/full-ibm_fez-20261025T094010Z/evaluation.json artifacts/benchmark/qpu-runs/full-ibm_fez-20261025T094010Z/test-kernel-raw.npy artifacts/benchmark/qpu-runs/full-ibm_fez-20261025T094010Z/test-kernel-calibrated.npy
Wrote: artifacts/benchmark/qpu-runs/pilot-ibm_fez-20261025T092140Z/evaluation.json


,"IBM hardware, recomputed from counts"
circuits,336
shots returned per circuit,[1024]
diagonal controls mean (ideal 1),0.924357
calibration slope a,0.819891
calibration intercept c,0.017931
calibration RMSE,0.025625
calibrated test values clipped (of 288),79
"test kernel error vs exact, uncorrected",0.037319
"test kernel error vs exact, calibrated",0.029469
"error vs exact over all 336 circuits, uncorrected",0.040055


## 3. Classification from the hardware kernels

The trained quantum-kernel SVM, fitted on the exact simulated training kernel, classifies the 12 test examples from three test kernels: the exact simulated one (from notebook 03), the uncorrected hardware one, and the calibrated hardware one. Nothing is refitted or tuned here.

In [7]:
simulation = read(qpu_run.SIMULATION_EVALUATION)
assert simulation["inputs"] == data.input_hashes
evaluations = {
    "exact simulation": simulation["evaluations"]["quantum_kernel_svm"],
    "IBM hardware, uncorrected": full["evaluations"]["qpu"],
    "IBM hardware, calibrated": full["evaluations"]["qpu_calibrated"],
}
pd.DataFrame(
    {
        source: {
            **{name: round(value, 3) for name, value in e["metrics"].items()},
            "TN / FP / FN / TP": " / ".join(str(v) for row in e["confusion_matrix"] for v in row),
        }
        for source, e in evaluations.items()
    }
).T

,accuracy,balanced_accuracy,precision,recall,f1,average_precision,TN / FP / FN / TP
exact simulation,0.5,0.5,0.5,0.667,0.571,0.526,2 / 4 / 2 / 4
"IBM hardware, uncorrected",0.5,0.5,0.5,0.667,0.571,0.538,2 / 4 / 2 / 4
"IBM hardware, calibrated",0.5,0.5,0.5,0.667,0.571,0.509,2 / 4 / 2 / 4


In [8]:
# Replication check: recomputed hardware rows equal the published rows.
published = read("reports/benchmark-results.json")
rows = {(row["name"], row["kernel_source"]): row for row in published["models"]}
for source in ("qpu", "qpu_calibrated"):
    row, mine = rows[("Quantum-kernel SVM", source)], full["evaluations"][source]
    assert row["confusion_matrix"] == mine["confusion_matrix"], source
    assert row["metrics"].keys() == mine["metrics"].keys()
    for name, value in row["metrics"].items():
        assert abs(value - mine["metrics"][name]) <= TOL, (
            source,
            name,
            value,
            mine["metrics"][name],
        )
    print(
        f"OK: {source:<14} metrics (1e-9) and confusion matrix {mine['confusion_matrix']} equal reports/benchmark-results.json"
    )
assert published["run_id"] == FULL_RUN and published["status"] == "complete"
assert published["hardware"] == {"backend": "ibm_fez", "shots": 1024, "job_ids": [job["job_id"]]}
qpu_usage = published["runtime"]["qpu"]
assert abs(qpu_usage["execution_seconds"] - full["usage"]["execution_seconds"]) <= TOL
assert qpu_usage["charged_seconds"] == full["usage"]["charged_seconds"]
assert abs(qpu_usage["resource_units"] - full["usage"]["resource_units"]) <= TOL
print("OK: published run_id, backend, shots, job ID and QPU usage equal the recorded run")
# Classical accuracies quoted in the Interpretation section below.
assert f"{rows[('Logistic regression', 'classical')]['metrics']['accuracy']:.3f}" == "0.833"
assert f"{rows[('RBF SVM', 'classical')]['metrics']['accuracy']:.3f}" == "0.917"
print("OK: published classical accuracies are 0.833 (logistic regression) and 0.917 (RBF SVM)")

OK: qpu            metrics (1e-9) and confusion matrix [[2, 4], [2, 4]] equal reports/benchmark-results.json
OK: qpu_calibrated metrics (1e-9) and confusion matrix [[2, 4], [2, 4]] equal reports/benchmark-results.json
OK: published run_id, backend, shots, job ID and QPU usage equal the recorded run
OK: published classical accuracies are 0.833 (logistic regression) and 0.917 (RBF SVM)


In [9]:
# Replication check: the hardware measures quoted in docs/benchmark-protocol.md.
sim_scores = np.array(evaluations["exact simulation"]["decision_scores"])
correlation = {
    source: float(np.corrcoef(sim_scores, full["evaluations"][source]["decision_scores"])[0, 1])
    for source in ("qpu", "qpu_calibrated")
}
protocol = Path("docs/benchmark-protocol.md").read_text(encoding="utf-8")
checks = [
    ("calibration slope", f"{cal['a']:.3f}", "0.820"),
    ("calibration intercept", f"{cal['c']:.3f}", "0.018"),
    ("calibration RMSE", f"{cal['rmse']:.3f}", "0.026"),
    ("kernel error, uncorrected", f"{full['test_kernel_error_vs_ideal']['qpu']:.3f}", "0.037"),
    (
        "kernel error, calibrated",
        f"{full['test_kernel_error_vs_ideal']['qpu_calibrated']:.3f}",
        "0.029",
    ),
    ("clipped calibrated values", str(cal["clipped_test_values"]), "79"),
    ("diagonal controls mean", f"{full['diagonal_controls_mean']:.3f}", "0.924"),
    ("score correlation, uncorrected", f"{correlation['qpu']:.3f}", "0.992"),
    ("score correlation, calibrated", f"{correlation['qpu_calibrated']:.3f}", "0.985"),
]
for name, recomputed, documented in checks:
    assert recomputed == documented, f"{name}: recomputed {recomputed}, protocol says {documented}"
    print(f"OK: {name:<31} {recomputed} (protocol {documented})")
quoted = [
    "slope 0.820, intercept 0.018, RMSE 0.026",
    "0.037 uncorrected, 0.029 calibrated",
    "79 of 288",
    "mean 0.924, ideal 1",
    "0.992 uncorrected, 0.985 calibrated",
]
assert all(text in protocol for text in quoted), "the protocol text no longer quotes these values"
assert full["shots_returned"] == [1024]
for source in ("qpu", "qpu_calibrated"):
    assert (
        full["evaluations"][source]["predictions"] == evaluations["exact simulation"]["predictions"]
    )
print("OK: the protocol quotes exactly these values")
print("OK: every circuit returned 1,024 shots")
print(
    "OK: uncorrected and calibrated hardware predictions are identical to the exact-simulation predictions"
)

OK: calibration slope               0.820 (protocol 0.820)
OK: calibration intercept           0.018 (protocol 0.018)
OK: calibration RMSE                0.026 (protocol 0.026)
OK: kernel error, uncorrected       0.037 (protocol 0.037)
OK: kernel error, calibrated        0.029 (protocol 0.029)
OK: clipped calibrated values       79 (protocol 79)
OK: diagonal controls mean          0.924 (protocol 0.924)
OK: score correlation, uncorrected  0.992 (protocol 0.992)
OK: score correlation, calibrated   0.985 (protocol 0.985)
OK: the protocol quotes exactly these values
OK: every circuit returned 1,024 shots
OK: uncorrected and calibrated hardware predictions are identical to the exact-simulation predictions


## 4. Raw, calibrated, and ideal kernels

Hardware noise pulls measured kernel values into a smaller, flatter range. The calibration is a straight-line fit `measured = a * ideal + c`, by least squares on the 24 **training** calibration pairs only; their exact values are known, and test labels and test kernel values never enter the fit. The fit is then inverted and applied to the test kernel: `calibrated = clip((measured - c) / a, 0, 1)`. A value is clipped when noise places it below the intercept `c` (or above `a + c`).

In [10]:
ideal_test = qkernel.statevector_kernel(data.test_angles, data.train_angles)
raw = np.load(full_dir / "test-kernel-raw.npy")
calibrated = np.load(full_dir / "test-kernel-calibrated.npy")

# Rebuild the calibration from the recorded counts with the project helpers.
plan = [hardware.PairSpec(**spec) for spec in full_record["plan"]]
counts = qpu_run.collected_counts(full_dir, full_record)
assembled = hardware.assemble(plan, counts, len(data.train_angles), len(data.test_angles))
ideal_train = qkernel.statevector_kernel(data.train_angles)
cal_ideal = np.array([ideal_train[i, j] for i, j, _ in assembled["calibration"]])
cal_measured = np.array([v for _, _, v in assembled["calibration"]])
refit = hardware.fit_calibration(cal_ideal, cal_measured)
remapped, reclipped = hardware.apply_calibration(raw, refit)

summary = (
    pd.DataFrame(
        {
            "exact simulation": ideal_test.ravel(),
            "IBM hardware, uncorrected": raw.ravel(),
            "IBM hardware, calibrated": calibrated.ravel(),
        }
    )
    .describe()
    .T[["mean", "std", "min", "50%", "max"]]
)
summary["mean |K - exact|"] = [
    0.0,
    np.abs(raw - ideal_test).mean(),
    np.abs(calibrated - ideal_test).mean(),
]
print("Test kernel, 12 x 24 = 288 values:")
summary.round(3)

Test kernel, 12 x 24 = 288 values:


,mean,std,min,50%,max,mean |K - exact|
exact simulation,0.209,0.298,0.000,0.046,1.00,0.000
"IBM hardware, uncorrected",0.199,0.262,0.003,0.064,0.93,0.037
"IBM hardware, calibrated",0.219,0.308,0.000,0.057,1.00,0.029


In [11]:
# A few entries: test example 0 against the first six training examples, and the calibration pairs.
display(
    pd.DataFrame(
        {
            "exact simulation": ideal_test[0, :6],
            "IBM hardware, uncorrected": raw[0, :6],
            "IBM hardware, calibrated": calibrated[0, :6],
        },
        index=[f"test 0 / train {j}" for j in range(6)],
    ).round(4)
)
pd.DataFrame(
    {
        "exact simulation": cal_ideal,
        "IBM hardware, measured": cal_measured,
        "fit a * ideal + c": refit["a"] * cal_ideal + refit["c"],
    },
    index=[f"train {i} / train {j}" for i, j, _ in assembled["calibration"]],
).round(4).head(8)

,exact simulation,"IBM hardware, uncorrected","IBM hardware, calibrated"
test 0 / train 0,0.6370,0.5957,0.7047
test 0 / train 1,0.4929,0.4639,0.5439
test 0 / train 2,1.0000,0.8955,1.0000
test 0 / train 3,0.7417,0.6846,0.8131
test 0 / train 4,0.0001,0.0156,0.0000
test 0 / train 5,0.2492,0.2588,0.2938


,exact simulation,"IBM hardware, measured",fit a * ideal + c
train 0 / train 23,0.0865,0.1143,0.0889
train 1 / train 3,0.9317,0.8184,0.7818
train 1 / train 13,0.0056,0.0205,0.0225
train 2 / train 8,0.2440,0.2197,0.2180
train 2 / train 10,0.2950,0.2061,0.2598
train 4 / train 20,0.2336,0.1846,0.2095
train 5 / train 12,0.3987,0.3506,0.3448
train 5 / train 13,0.0025,0.0264,0.0200


In [12]:
# Replication check: kernels and calibration rebuilt from the raw counts file WITHOUT the
# project's counts-to-kernel helpers (assemble, kernel_from_counts, fit_calibration,
# apply_calibration), so a bug in that code path would show up here.
saved_ideal = np.load(benchmark_inputs.ARTIFACT_DIR / "test-kernel-ideal.npy")
assert np.allclose(ideal_test, saved_ideal, rtol=0, atol=TOL)

raw_counts = []
for entry in sorted(full_record["jobs"], key=lambda job: job["start"]):
    raw_counts += read(full_dir / f"counts-{entry['job_id']}.json")
assert len(raw_counts) == len(full_record["plan"]) == 336
manual_raw = np.full((len(data.test_angles), len(data.train_angles)), np.nan)
manual_cal_pairs = []
for spec, shot_counts in zip(full_record["plan"], raw_counts, strict=True):
    p_zero = shot_counts.get("0000", 0) / sum(shot_counts.values())  # K = P(all zeros)
    if spec["kind"] == "test":
        manual_raw[spec["left"], spec["right"]] = p_zero
    elif spec["kind"] == "calibration":
        manual_cal_pairs.append((spec["left"], spec["right"], p_zero))
assert not np.isnan(manual_raw).any() and len(manual_cal_pairs) == 24
assert np.array_equal(raw, manual_raw)

manual_ideal = np.array([ideal_train[i, j] for i, j, _ in manual_cal_pairs])
manual_measured = np.array([v for _, _, v in manual_cal_pairs])
manual_a, manual_c = np.polyfit(manual_ideal, manual_measured, 1)
assert abs(manual_a - cal["a"]) <= TOL and abs(manual_c - cal["c"]) <= TOL
manual_rmse = np.sqrt(np.mean((manual_measured - (manual_a * manual_ideal + manual_c)) ** 2))
assert abs(manual_rmse - cal["rmse"]) <= TOL
manual_mapped = (manual_raw - manual_c) / manual_a
assert np.allclose(np.clip(manual_mapped, 0, 1), calibrated, rtol=0, atol=TOL)
manual_clipped = int(((manual_mapped < 0) | (manual_mapped > 1)).sum())
assert manual_clipped == cal["clipped_test_values"] == 79
assert calibrated.min() >= 0 and calibrated.max() <= 1
low = float((ideal_test < 0.02).mean())
assert round(100 * low) == 37, low
print("OK: exact test kernel equals artifacts/benchmark/test-kernel-ideal.npy (1e-9)")
print(
    "OK: test-kernel-raw.npy equals P('0000') read directly from the recorded counts file (exact)"
)
print(
    f"OK: independent np.polyfit a={manual_a:.6f}, c={manual_c:.6f} and RMSE equal evaluate_run's calibration (1e-9)"
)
print("OK: test-kernel-calibrated.npy equals clip((raw - c) / a, 0, 1) (1e-9); 79 of 288 clipped")
print(f"OK: {100 * low:.0f}% of exact test values are below 0.02, as the protocol states")

OK: exact test kernel equals artifacts/benchmark/test-kernel-ideal.npy (1e-9)
OK: test-kernel-raw.npy equals P('0000') read directly from the recorded counts file (exact)
OK: independent np.polyfit a=0.819891, c=0.017931 and RMSE equal evaluate_run's calibration (1e-9)
OK: test-kernel-calibrated.npy equals clip((raw - c) / a, 0, 1) (1e-9); 79 of 288 clipped
OK: 37% of exact test values are below 0.02, as the protocol states


## 5. The pilot and its projection

Before the full run, a 12-circuit pilot (the first 12 test pairs) measured what the workload costs. `evaluate_run` on a pilot reports only usage and kernel error, and projects the full 336-circuit run linearly per circuit. Pilot results are never published as benchmark results.

In [13]:
print(
    f"Pilot {pilot['run_id']}: {pilot['circuits']} circuits, shots returned {pilot['shots_returned']}, "
    f"mean |kernel - exact| {pilot['mean_abs_kernel_error_vs_ideal']:.3f} (IBM hardware)"
)
pd.DataFrame(
    {
        "pilot, 12 circuits (IBM hardware)": pilot["usage"],
        "projected, 336 circuits": pilot["projected_full_run"],
        "measured full run, 336 circuits (IBM hardware)": full["usage"],
    }
).round(3)

Pilot pilot-ibm_fez-20261025T092140Z: 12 circuits, shots returned [1024], mean |kernel - exact| 0.034 (IBM hardware)


,"pilot, 12 circuits (IBM hardware)","projected, 336 circuits","measured full run, 336 circuits (IBM hardware)"
charged_seconds,5.000,140.000,92.000
resource_units,0.082,2.289,1.526
execution_seconds,3.181,89.072,89.187


In [14]:
# Replication check: pilot numbers in docs/benchmark-protocol.md.
p, proj = pilot["usage"], pilot["projected_full_run"]
assert pilot["pilot"] and pilot["circuits"] == 12 and pilot["shots_returned"] == [1024]
assert "evaluations" not in pilot
assert (
    p["charged_seconds"] == 5
    and round(p["execution_seconds"], 1) == 3.2
    and round(p["resource_units"], 3) == 0.082
)
assert (
    round(proj["execution_seconds"]) == 89
    and proj["charged_seconds"] == 140
    and round(proj["resource_units"], 1) == 2.3
)
assert f"{pilot['mean_abs_kernel_error_vs_ideal']:.3f}" == "0.034"
pilot_quoted = [
    "| 3.2 s | about 89 s |",
    "| 5 s | up to 140 s",
    "| 0.082 | about 2.3 of the 10 available |",
    "mean absolute kernel error against the exact kernel was 0.034",
]
assert all(text in protocol for text in pilot_quoted), (
    "the protocol's pilot table no longer quotes these values"
)
print("OK: the protocol's pilot table quotes exactly these values")
print(
    "OK: pilot 3.2 s execution, 5 s billed, 0.082 resource units; projection about 89 s, up to 140 s, about 2.3 units"
)
print("OK: pilot mean kernel error 0.034")
print(
    f"Measured full run: {full['usage']['execution_seconds']:.1f} s execution vs {proj['execution_seconds']:.1f} s projected; "
    f"billed {full['usage']['charged_seconds']:.0f} s vs {proj['charged_seconds']:.0f} s projected (billing is rounded per job)"
)

OK: the protocol's pilot table quotes exactly these values
OK: pilot 3.2 s execution, 5 s billed, 0.082 resource units; projection about 89 s, up to 140 s, about 2.3 units
OK: pilot mean kernel error 0.034
Measured full run: 89.2 s execution vs 89.1 s projected; billed 92 s vs 140 s projected (billing is rounded per job)


## 6. The replay file

The web demo does not run the quantum-kernel SVM live, because that would need new QPU jobs. It shows **replay**: the recorded per-sample test predictions in `reports/benchmark-replay.json`. Samples are identified only by test position, dataset `row_index`, and hashed group, never by host or URL. Here the replay is rebuilt with `src.evaluation.replay.build` from the split manifest, the simulation evaluation, and the evaluation just recomputed from the counts, then compared with the published file.

In [15]:
manifest = read(benchmark_inputs.MANIFEST_PATH)
qpu = {
    "run_id": full_record["run_id"],
    "backend": full_record["backend"],
    "inputs": full_record["inputs"],
    "evaluations": full["evaluations"],
}
rebuilt = replay.build(manifest, simulation, qpu)
published_replay = read("reports/benchmark-replay.json")

pd.DataFrame(
    [
        {
            "position": s["position"],
            "row_index": s["row_index"],
            "group_hash": s["group_hash"],
            "label": "phishing" if s["is_phishing"] else "legitimate",
            **{source: s["predictions"][source]["prediction"] for source in replay.SOURCES},
        }
        for s in rebuilt["samples"]
    ]
).set_index("position")

,row_index,group_hash,label,simulation,qpu,qpu_calibrated
position,,,,,,
0,15517,fd393998db75545c,legitimate,phishing,phishing,phishing
1,174792,b2d1a9ba5000a09a,legitimate,legitimate,legitimate,legitimate
2,217648,84d275114cab3730,legitimate,legitimate,legitimate,legitimate
3,161204,c660c6c6a7b6fffe,legitimate,phishing,phishing,phishing
4,94665,010ccedeafebd986,legitimate,phishing,phishing,phishing
5,18899,9a021f91155117e4,legitimate,phishing,phishing,phishing
6,219334,55624b9e9738e434,phishing,legitimate,legitimate,legitimate
7,41538,e51ce3301401094b,phishing,legitimate,legitimate,legitimate
8,121258,62c0d81c126f3930,phishing,phishing,phishing,phishing


In [16]:
# Replication check: the published replay equals the rebuilt one (generation timestamp aside).
assert (
    published_replay["source_runs"]
    == rebuilt["source_runs"]
    == {"simulation": simulation["run_id"], "qpu": FULL_RUN, "backend": "ibm_fez"}
)


def header(r):
    return {k: v for k, v in r.items() if k not in ("samples", "generated_at_utc")}


assert header(published_replay) == header(rebuilt)
assert len(published_replay["samples"]) == len(rebuilt["samples"]) == 12
for mine, theirs in zip(rebuilt["samples"], published_replay["samples"], strict=True):
    assert set(theirs) == {"position", "row_index", "group_hash", "is_phishing", "predictions"}, (
        "unexpected sample fields"
    )
    for key in ("position", "row_index", "group_hash", "is_phishing"):
        assert mine[key] == theirs[key], key
    assert mine["predictions"].keys() == theirs["predictions"].keys() == set(replay.SOURCES)
    for source in replay.SOURCES:
        a, b = mine["predictions"][source], theirs["predictions"][source]
        assert (a["prediction"], a["is_phishing"]) == (b["prediction"], b["is_phishing"]), (
            mine["position"],
            source,
        )
        assert abs(a["decision_score"] - b["decision_score"]) <= TOL, (mine["position"], source)
print(
    "OK: reports/benchmark-replay.json has the same 12 samples, labels, predictions and decision scores (1e-9) for simulation, qpu, qpu_calibrated"
)
print("OK: replay samples carry only position, row_index, group_hash, label and predictions")

OK: reports/benchmark-replay.json has the same 12 samples, labels, predictions and decision scores (1e-9) for simulation, qpu, qpu_calibrated
OK: replay samples carry only position, row_index, group_hash, label and predictions


## 7. Interpretation

- **The hardware reproduced the simulated classifier.** On `ibm_fez` the test kernel deviates from the exact simulation by 0.037 on average (0.029 after calibration), and the calibration slope of 0.820 shows that noise shrinks kernel values. Yet both hardware variants give exactly the same 12 predictions as the exact simulation, and the SVM decision scores correlate at 0.992 (uncorrected) and 0.985 (calibrated) with the simulated ones. Only average precision moves (0.526 exact simulation, 0.538 uncorrected, 0.509 calibrated), because it depends on how the scores rank the examples.
- **The result is chance level, and that comes from the model, not the hardware.** Accuracy and balanced accuracy are 0.500 for all three kernels, against 0.833 for logistic regression and 0.917 for the RBF SVM (classical, notebook 03). The four coarse host features and this kernel do not separate the classes on this split.
- **Calibration is a training-only correction**, fixed before any test result was seen, and nothing here is tuned on test results. It reduced the kernel error by about a fifth but clipped 79 of 288 values, mostly ideal values near zero that noise placed below the intercept.
- **Limitations.** With 12 test examples, each example moves accuracy by about 8.3 percentage points. There is one split and one seed, so no variability estimate exists. The SVM was trained on an exact kernel and only tested on hardware kernels; no model trained on hardware measurements is evaluated. Four qubits are trivial to simulate, so the 89.2 s of hardware execution against milliseconds of simulation says nothing about larger problems. **No quantum advantage is claimed.**

## 8. Optional: noisy rehearsal on a local fake backend

Before using real quota, the same 336-circuit workload ran on `FakeFez`, a local fake backend with the noise model and connectivity of a 156-qubit IBM Heron device. This is **noisy simulation, not hardware**, and needs no account. The documented command is `python scripts/rehearse_hardware.py`. It writes `artifacts/benchmark/rehearsal-fake_fez/`, which feeds the results chart. To leave that recorded rehearsal untouched, the cell below mirrors the script's `main()` step for step in memory and writes nothing; it first checks that the script's defaults (`FakeFez`, sampler `v2`) and project calls still match. It takes about 15 s; set `RUN_REHEARSAL = False` to skip it.

What is exact and what is not: the pair plan and the transpilation are seeded (seed 42), so the transpiled circuits and their statistics are deterministic. The simulator's shot sampling is not seeded, so counts, the calibration fit, and kernel errors change slightly on every run. They are compared with the protocol's rehearsal table only within shot-noise tolerances.

In [17]:
RUN_REHEARSAL = True

if RUN_REHEARSAL:
    import ast

    from qiskit_ibm_runtime.fake_provider import FakeFez

    # This cell mirrors scripts/rehearse_hardware.py main() step for step, in memory and
    # without writing its outputs. Tie it to the script: same defaults, same project calls.
    script_src = Path("scripts/rehearse_hardware.py").read_text(encoding="utf-8")
    defaults = {
        call.args[0].value: kw.value.value
        for call in ast.walk(ast.parse(script_src))
        if isinstance(call, ast.Call) and getattr(call.func, "attr", "") == "add_argument"
        for kw in call.keywords
        if kw.arg == "default"
    }
    assert defaults == {"--backend": "FakeFez", "--sampler": "v2"}, defaults
    assert "v2" in hardware.SAMPLERS
    mirrored = [
        "hardware.plan_pairs(",
        "hardware.build_circuits(",
        "hardware.transpile_for(",
        "hardware.run_sampler(",
        "hardware.assemble(",
        "hardware.fit_calibration(",
        "hardware.apply_calibration(",
        "qkernel.statevector_kernel(",
        "evaluate(model,",
    ]
    assert all(step in script_src for step in mirrored), (
        "rehearse_hardware.py changed; update this cell"
    )
    print(
        "OK: rehearse_hardware.py defaults (FakeFez, sampler v2) and project calls match this cell"
    )

    backend = FakeFez()
    n_train, n_test = len(data.train_angles), len(data.test_angles)
    fake_plan = hardware.plan_pairs(n_train, n_test)
    circuits = hardware.build_circuits(fake_plan, data.train_angles, data.test_angles)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)  # SamplerV2 deprecation notice
        isa, fake_stats = hardware.transpile_for(backend, circuits)
        fake_counts, fake_job = hardware.run_sampler(backend, isa, implementation="v2")
    fake = hardware.assemble(fake_plan, fake_counts, n_train, n_test)
    fake_cal = hardware.fit_calibration(
        np.array([ideal_train[i, j] for i, j, _ in fake["calibration"]]),
        np.array([v for _, _, v in fake["calibration"]]),
    )
    fake_calibrated, fake_clipped = hardware.apply_calibration(fake["test_kernel"], fake_cal)
    fake_error = {
        "raw": float(np.abs(fake["test_kernel"] - ideal_test).mean()),
        "calibrated": float(np.abs(fake_calibrated - ideal_test).mean()),
    }
    fake_diagonal = float(np.mean([v for _, v in fake["diagonal"]]))
    model = data.models["quantum_kernel_svm"]
    fake_eval = {
        "noisy raw": evaluate(model, fake["test_kernel"], data.test_labels),
        "noisy calibrated": evaluate(model, fake_calibrated, data.test_labels),
    }
    display(
        pd.DataFrame(
            {
                "noisy simulation, FakeFez (this run)": {
                    "depth min-max": f"{fake_stats['depth']['min']}-{fake_stats['depth']['max']}",
                    "two-qubit gates min-max": f"{fake_stats['two_qubit_gates']['min']}-{fake_stats['two_qubit_gates']['max']}",
                    "diagonal controls mean": f"{fake_diagonal:.3f}",
                    "calibration slope / intercept": f"{fake_cal['a']:.3f} / {fake_cal['c']:.3f}",
                    "kernel error raw / calibrated": f"{fake_error['raw']:.3f} / {fake_error['calibrated']:.3f}",
                    "predictions vs exact simulation": "identical"
                    if all(
                        e["predictions"] == evaluations["exact simulation"]["predictions"]
                        for e in fake_eval.values()
                    )
                    else "differ",
                },
                "noisy simulation, protocol table": {
                    "depth min-max": "1-54",
                    "two-qubit gates min-max": "0-10",
                    "diagonal controls mean": "0.982",
                    "calibration slope / intercept": "0.960 / 0.007",
                    "kernel error raw / calibrated": "0.013 / 0.009",
                    "predictions vs exact simulation": "identical",
                },
                "IBM hardware, ibm_fez (recorded)": {
                    "depth min-max": f"{t['depth']['min']}-{t['depth']['max']}",
                    "two-qubit gates min-max": f"{t['two_qubit_gates']['min']}-{t['two_qubit_gates']['max']}",
                    "diagonal controls mean": f"{full['diagonal_controls_mean']:.3f}",
                    "calibration slope / intercept": f"{cal['a']:.3f} / {cal['c']:.3f}",
                    "kernel error raw / calibrated": f"{full['test_kernel_error_vs_ideal']['qpu']:.3f} / {full['test_kernel_error_vs_ideal']['qpu_calibrated']:.3f}",
                    "predictions vs exact simulation": "identical",
                },
            }
        )
    )
    print(
        f"Local sampling took {fake_job['client_seconds']:.1f} s (noisy simulation; not QPU time)."
    )

OK: rehearse_hardware.py defaults (FakeFez, sampler v2) and project calls match this cell


,"noisy simulation, FakeFez (this run)","noisy simulation, protocol table","IBM hardware, ibm_fez (recorded)"
depth min-max,1-54,1-54,1-54
two-qubit gates min-max,0-10,0-10,0-10
diagonal controls mean,0.983,0.982,0.924
calibration slope / intercept,0.925 / 0.014,0.960 / 0.007,0.820 / 0.018
kernel error raw / calibrated,0.013 / 0.009,0.013 / 0.009,0.037 / 0.029
predictions vs exact simulation,identical,identical,identical


Local sampling took 10.4 s (noisy simulation; not QPU time).


In [18]:
# Replication check (loose): rehearsal. Exact for the seeded parts, tolerances for shot noise.
if RUN_REHEARSAL:
    assert [vars(s) for s in fake_plan] == full_record["plan"]
    assert (fake_stats["depth"]["min"], fake_stats["depth"]["max"]) == (1, 54)
    assert (fake_stats["two_qubit_gates"]["min"], fake_stats["two_qubit_gates"]["max"]) == (0, 10)
    assert fake_job["shots_returned"] == [1024] and len(fake_counts) == 336
    rehearsal_quoted = [
        "| Transpiled depth / two-qubit gates | 1 to 54 / 0 to 10 per circuit |",
        "| Diagonal controls | mean 0.982, ideal 1 |",
        "slope 0.960, intercept 0.007",
        "| Mean absolute kernel error vs ideal | 0.013 raw, 0.009 calibrated |",
        "same predictions as the ideal kernel",
    ]
    assert all(text in protocol for text in rehearsal_quoted), (
        "the protocol's rehearsal table changed"
    )
    assert abs(fake_diagonal - 0.982) < 0.01
    assert (
        fake_cal["valid"]
        and abs(fake_cal["a"] - 0.960) < 0.06
        and abs(fake_cal["c"] - 0.007) < 0.01
    )
    assert abs(fake_error["raw"] - 0.013) < 0.003 and abs(fake_error["calibrated"] - 0.009) < 0.003
    for name, e in fake_eval.items():
        assert e["predictions"] == evaluations["exact simulation"]["predictions"], name
    same_stats = all(
        fake_stats[k] == t[k] for k in ("depth", "two_qubit_gates", "distinct_physical_layouts")
    )
    print("OK: the reference column above is the protocol's rehearsal table, quoted verbatim")
    print(
        "OK: rehearsal plan equals the recorded hardware plan; depth 1-54 and two-qubit gates 0-10 as in the protocol (deterministic)"
    )
    print(
        "OK: diagonal mean, calibration and kernel errors within shot-noise tolerance of the protocol's rehearsal table"
    )
    print("OK: noisy raw and calibrated predictions equal the exact-simulation predictions")
    print(f"Transpilation statistics identical to the recorded ibm_fez run: {same_stats}")
    print(
        f"The hardware kernel error is {full['test_kernel_error_vs_ideal']['qpu'] / fake_error['raw']:.1f} times this rehearsal's: real noise was larger than the fake backend suggested."
    )
else:
    print("Rehearsal skipped.")

OK: the reference column above is the protocol's rehearsal table, quoted verbatim
OK: rehearsal plan equals the recorded hardware plan; depth 1-54 and two-qubit gates 0-10 as in the protocol (deterministic)
OK: diagonal mean, calibration and kernel errors within shot-noise tolerance of the protocol's rehearsal table
OK: noisy raw and calibrated predictions equal the exact-simulation predictions
Transpilation statistics identical to the recorded ibm_fez run: True
The hardware kernel error is 2.9 times this rehearsal's: real noise was larger than the fake backend suggested.


## 9. Running new hardware jobs (not executed here)

Everything above used recorded counts. New jobs need an **IBM Quantum Open Plan account** and use real QPU quota. Copy `.env.example` to `.env` and set `IBM_QUANTUM_TOKEN`, `IBM_QUANTUM_INSTANCE`, and `IBM_QUANTUM_BACKEND` (for example `ibm_fez`). `.env` is ignored by Git and must never be committed. The documented sequence, from the repository root with the project environment:

```powershell
python scripts/check_ibm_access.py                   # account, instance, quota and backend access
python scripts/run_qpu.py submit --pilot 12          # 12-circuit pilot; asks you to type SUBMIT
python scripts/run_qpu.py retrieve <pilot-run-id>    # repeatable; never resubmits
python scripts/run_qpu.py evaluate <pilot-run-id>    # local: usage and full-run projection
python scripts/run_qpu.py submit                     # full 336-circuit run, if the projection fits the quota
python scripts/run_qpu.py retrieve <full-run-id>
python scripts/run_qpu.py evaluate <full-run-id>     # local: rewrites reports/benchmark-results.json
python scripts/publish_replay.py                     # local: rewrites reports/benchmark-replay.json
python scripts/time_benchmark.py                     # local: adds runtimes back to the results file
python scripts/render_results_svg.py                 # local: regenerates docs/benchmark-results.svg
```

Cost: the recorded full run billed 92 s of the Open Plan's 600 s per rolling 28-day window, and the pilot 5 s. A new run produces new counts, with new shot noise and possibly a different device calibration, so its kernel errors, calibration fit, and possibly its predictions will differ slightly from the recorded results. The replication checks in this notebook apply only to the versioned evidence.